# Settings

## Import Packages

In [11]:
import osmnx as ox
import pandas as pd
import numpy as np
import os
import re
from scipy import stats
from scipy.stats import chi2_contingency, pointbiserialr

import json
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import math
import itertools
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import Point
import requests
import time
from geopy.geocoders import Nominatim
import time
from shapely.wkt import dumps
from shapely.wkt import loads
from dotenv import load_dotenv
import glob

pd.set_option('display.precision', 4)

## Self-defined functions

### save_geoDataFrame

In [12]:
def save_geoDataFrame(source_df, filename):
    """
    Save a GeoDataFrame to a CSV file by converting the geometry column to WKT format.

    Parameters:
        source_df (GeoDataFrame): The GeoDataFrame to be saved.
        filename (str): Path to the output CSV file.

    Notes:
        - The geometry column will be converted to WKT strings using shapely.wkt.dumps().
        - The resulting CSV can be restored into a GeoDataFrame using shapely.wkt.loads().
    """
    df = source_df.copy()
    df['geometry'] = df['geometry'].apply(dumps)
    df.to_csv(filename, index=False)

### load_geoDataFrame

In [13]:
def load_geoDataFrame(filepath):
    """
    Load a GeoDataFrame from a CSV file with a geometry column in WKT format.

    Parameters:
        filepath (str): Path to the CSV file containing the saved GeoDataFrame.

    Returns:
        GeoDataFrame: A GeoDataFrame reconstructed from the CSV, with geometries parsed from WKT strings.
    
    Notes:
        - Assumes the geometry column is named 'geometry' and stored in WKT format.
        - The returned GeoDataFrame is assigned the CRS 'EPSG:4326' by default.
    """
    df = pd.read_csv(filepath)
    df['geometry'] = df['geometry'].apply(loads)
    return gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

### gen_point_area_mapping

In [14]:
def gen_point_area_mapping(point_df, lat_col, lon_col, geometry_df, geometry_col='geometry'):
    """
    Maps each point in point_df to a polygon in geometry_df using spatial join.

    Parameters:
        point_df (pd.DataFrame): DataFrame with latitude and longitude columns.
        lat_col (str): Column name for latitude.
        lon_col (str): Column name for longitude.
        geometry_df (gpd.GeoDataFrame): GeoDataFrame containing polygon geometries.
        geometry_col (str): Column name for the geometry column in geometry_df.

    Returns:
        gpd.GeoDataFrame: Result of spatial join with matching polygons.
    """
    # Drop rows with missing coordinates
    point_df = point_df.dropna(subset=[lat_col, lon_col]).copy()

    # Convert to GeoDataFrame
    point_gdf = gpd.GeoDataFrame(
        point_df,
        geometry=gpd.points_from_xy(point_df[lon_col], point_df[lat_col]),
        crs="EPSG:4326"
    )

    # Ensure geometry_df is a GeoDataFrame with proper CRS
    area_gdf = geometry_df.copy()
    if geometry_col != 'geometry':
        area_gdf = area_gdf.set_geometry(geometry_col)
    if area_gdf.crs != "EPSG:4326":
        area_gdf = area_gdf.to_crs("EPSG:4326")

    # Perform spatial join
    joined = gpd.sjoin(point_gdf, area_gdf, how='inner', predicate='within')

    return joined

### gen_point_area_buffer_mapping

In [15]:
def gen_point_area_buffer_mapping(point_df, lat_col, lon_col, grid_df, grid_id_col='grid_id', buffer_radius_m=1000):
    """
    Maps each point in point_df to a nearby grid cell (from grid_df) using a centroid-based buffer.

    Parameters:
        point_df (pd.DataFrame): DataFrame with point locations, must include latitude and longitude columns.
        lat_col (str): Name of the latitude column in point_df.
        lon_col (str): Name of the longitude column in point_df.
        grid_df (gpd.GeoDataFrame): GeoDataFrame with polygon grid geometries (e.g., Manhattan grid).
        grid_id_col (str): Column in grid_df representing the unique ID of each grid (default: 'grid_id').
        buffer_radius_m (float): Radius in meters to buffer around each grid's centroid (default: 1000 meters).

    Returns:
        gpd.GeoDataFrame: A GeoDataFrame with all points that fall within any grid centroid's buffer,
                          including matched grid IDs.
    """
    # Project grid to a suitable projected CRS (NYC local)
    grid_proj = grid_df.to_crs(epsg=2263)
    grid_proj['centroid'] = grid_proj.geometry.centroid
    grid_proj['buffer'] = grid_proj['centroid'].buffer(buffer_radius_m)

    # Create buffer GeoDataFrame
    buffer_gdf = grid_proj[[grid_id_col, 'buffer']].copy()
    buffer_gdf = buffer_gdf.set_geometry('buffer')
    buffer_gdf = buffer_gdf.set_crs(epsg=2263)

    # Prepare point GeoDataFrame and project to same CRS
    point_df = point_df.dropna(subset=[lat_col, lon_col]).copy()
    point_gdf = gpd.GeoDataFrame(
        point_df,
        geometry=gpd.points_from_xy(point_df[lon_col], point_df[lat_col]),
        crs="EPSG:4326"
    )
    point_proj = point_gdf.to_crs(epsg=2263)

    # Drop potential naming conflict column
    if 'index_right' in point_proj.columns:
        point_proj = point_proj.drop(columns=['index_right'])

    # Spatial join: points within buffer zones
    joined = gpd.sjoin(point_proj, buffer_gdf, how='inner', predicate='within')

    return joined

### gen_area_area_mapping

In [16]:
def gen_area_area_mapping(zone_df, grid_df, zone_location_id):
    """
    Maps each polygon in zone_df to intersecting polygons in grid_df using spatial join.

    Parameters:
        zone_df (gpd.GeoDataFrame): GeoDataFrame with zone polygons (must include geometry).
        grid_df (gpd.GeoDataFrame): GeoDataFrame with grid polygons (must include 'grid_id' and geometry).
        zone_location_id (str): Column name in zone_df representing the zone identifier.

    Returns:
        gpd.GeoDataFrame: A DataFrame mapping each grid_id to zone_location_id for all intersecting areas.
                          The resulting dataframe includes ['grid_id', zone_location_id] and geometry columns.
    """
    # Ensure both inputs are GeoDataFrames
    zone_gdf = zone_df.copy()
    grid_gdf = grid_df.copy()

    zone_gdf = zone_gdf.set_geometry('geometry')
    grid_gdf = grid_gdf.set_geometry('geometry')

    # Reproject both to WGS84 if needed
    zone_gdf = zone_gdf.to_crs('EPSG:4326')
    grid_gdf = grid_gdf.to_crs('EPSG:4326')

    # Perform spatial join based on intersection
    zone_to_grid = gpd.sjoin(
        grid_gdf[['grid_id', 'geometry']],
        zone_gdf[[zone_location_id, 'geometry']],
        how='inner',
        predicate='intersects'
    )

    return zone_to_grid

### save_large_dataframe_in_chunks

In [17]:
def save_large_dataframe_in_chunks(source_df, file_name, chunk_size=1000000):
    """
    Saves a large GeoDataFrame into multiple CSV files in chunks to avoid memory issues.

    Parameters:
        source_df (GeoDataFrame): The GeoDataFrame to save.
        file_name (str): Base filename for the output files (without extension).
        chunk_size (int): Number of rows per chunk file. Default is 1,000,000.

    Returns:
        None
    """
    for i, start in enumerate(range(0, len(source_df), chunk_size)):
        end = start + chunk_size
        chunk = source_df.iloc[start:end]
        full_filepath = f"{file_name}_part{i+1}.csv"
        save_geoDataFrame(chunk, filename=full_filepath)
        print(full_filepath)

### load_all_geoDataFrames_in_folder

In [18]:
def load_all_geoDataFrames_in_folder(folder_path, pattern="*.csv"):
    """
    Load and concatenate all CSV files in a folder as GeoDataFrames.

    Parameters:
        folder_path (str): Path to the folder containing CSV files.
        pattern (str): Filename pattern to match CSV chunks (default: "*.csv").

    Returns:
        GeoDataFrame: A single concatenated GeoDataFrame with proper geometry column.
    """
    all_files = sorted(glob.glob(os.path.join(folder_path, pattern)))
    gdf_list = []

    for file in all_files:
        df = pd.read_csv(file)
        if 'geometry' in df.columns:
            df['geometry'] = df['geometry'].apply(loads)
            gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")
        else:
            raise ValueError(f"'geometry' column missing in {file}")
        gdf_list.append(gdf)
        print(file)

    return pd.concat(gdf_list, ignore_index=True)

# Load Datasets
- Some files are saved in the [shared Google Drive](https://drive.google.com/drive/u/2/folders/1Yr7l0EcolHcL2TDrrq72ZUix2FNEc5X2)
- Please download the files in the corresponding folder

In [19]:
base_path = '../data_preparation/'
grid_df = load_geoDataFrame(base_path + 'manhattan_grid/manhattan_grid.csv')
# Dataframe
weather_df = pd.read_csv(base_path + 'hourly_weather/hourly_weather_data_20240401_20250331.csv')
holiday_df = pd.read_csv(base_path + 'holiday_module/Holiday/holidays.csv')
inspection_df = pd.read_csv(base_path + 'inspection/Inspection_latest_Results_cleaned_step2.csv')
subway_df = pd.read_csv(base_path + 'mta_subway/mta_subway_20240101_20250430.csv')
wheelchair_df = pd.read_csv(base_path + 'wheelchair_accessibility/Wheelchair_Friendly/manhattan_wheelchair_friendly_restaurants.csv')

# GeoDataFrame
bike_df = load_geoDataFrame(base_path + 'citi_bike/citi_bike_20240401_20250331.csv')
event_df = load_geoDataFrame(base_path + 'event_data/event_data_20240401_20250331.csv')
population_df = load_geoDataFrame(base_path + 'population/census_data_2020.csv')
taxi_df = pd.read_csv(base_path + 'yellow_taxi/yellow_taxi_20240101_20250331.csv')
taxi_zones = load_geoDataFrame(base_path + 'manhattan_grid/taxi_zones.csv')

## Dataframe

### weather_df

In [20]:
integrated_df_1 = weather_df.merge(grid_df, how='cross')
integrated_df_1

,date,hour,temp_c,dew_c,wind_speed_knot,precip_mm,grid_id,lat,lon,geometry
0,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0001,40.6840,-74.0257,"POLYGON ((-74.0242 40.68292, -74.0242 40.68517..."
1,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0002,40.6840,-74.0227,"POLYGON ((-74.02126 40.68292, -74.02126 40.685..."
2,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0003,40.6840,-74.0198,"POLYGON ((-74.01832 40.68292, -74.01832 40.685..."
3,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0004,40.6863,-74.0257,"POLYGON ((-74.0242 40.68517, -74.0242 40.68742..."
4,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0005,40.6863,-74.0227,"POLYGON ((-74.02126 40.68517, -74.02126 40.687..."
...,...,...,...,...,...,...,...,...,...,...
9986395,2025-03-31,23,15.6,13.3,1.5,0.8,M-1136,40.8777,-73.9139,"POLYGON ((-73.91244 40.87661, -73.91244 40.878..."
9986396,2025-03-31,23,15.6,13.3,1.5,0.8,M-1137,40.8777,-73.9110,"POLYGON ((-73.90949 40.87661, -73.90949 40.878..."
9986397,2025-03-31,23,15.6,13.3,1.5,0.8,M-1138,40.8777,-73.9080,"POLYGON ((-73.90655 40.87661, -73.90655 40.878..."
9986398,2025-03-31,23,15.6,13.3,1.5,0.8,M-1139,40.8800,-73.9227,"POLYGON ((-73.92126 40.87886, -73.92126 40.881..."


### holiday_df

In [21]:
holiday_df['start_date'] = pd.to_datetime(holiday_df['start_date'])
holiday_df['end_date'] = pd.to_datetime(holiday_df['end_date'])

all_holidays = set()

for _, row in holiday_df.iterrows():
    date_range = pd.date_range(row['start_date'], row['end_date'])
    all_holidays.update(date_range.date)

In [22]:
integrated_df_1['date'] = pd.to_datetime(integrated_df_1['date']).dt.date
integrated_df_1['is_holiday'] = integrated_df_1['date'].isin(all_holidays)
integrated_df_1['is_holiday'].sum()

np.int64(273600)

### inspection_df

In [23]:
inspection_joined = gen_point_area_mapping(point_df=inspection_df,
                                           lat_col='Latitude',
                                           lon_col='Longitude',
                                           geometry_df=grid_df,
                                           geometry_col='geometry')
restaurant_counts = inspection_joined.groupby('grid_id').size().reset_index(name='restaurant_count')
integrated_df_2 = integrated_df_1.merge(restaurant_counts, on='grid_id', how='left')
integrated_df_2['restaurant_count'] = integrated_df_2['restaurant_count'].fillna(0)
integrated_df_2.shape

(9986400, 12)

### subway_df

In [24]:
subway_stations = subway_df[['latitude', 'longitude']].drop_duplicates()
subway_joined = gen_point_area_mapping(point_df=subway_stations,
                                           lat_col='latitude',
                                           lon_col='longitude',
                                           geometry_df=grid_df,
                                           geometry_col='geometry')
subway_joined

,latitude,longitude,geometry,index_right,grid_id,lat,lon
0,40.7021,-74.0137,POINT (-74.01366 40.70207),37,M-0038,40.7021,-74.0139
1,40.7031,-74.0130,POINT (-74.01299 40.70309),37,M-0038,40.7021,-74.0139
2,40.7048,-74.0141,POINT (-74.01407 40.70482),43,M-0044,40.7043,-74.0139
3,40.7065,-74.0111,POINT (-74.01106 40.70647),52,M-0053,40.7066,-74.0110
4,40.7068,-74.0091,POINT (-74.0091 40.70682),53,M-0054,40.7066,-74.0080
...,...,...,...,...,...,...,...
3005,40.7682,-73.9819,POINT (-73.98193 40.76825),494,M-0495,40.7674,-73.9816
8584,40.7373,-73.9968,POINT (-73.99679 40.73734),256,M-0257,40.7381,-73.9963
15617,40.7553,-73.9868,POINT (-73.98676 40.75529),390,M-0391,40.7561,-73.9874
15846,40.7188,-73.9999,POINT (-73.99989 40.7188),123,M-0124,40.7178,-73.9992


In [25]:
subway_buffer_joined = gen_point_area_buffer_mapping(point_df=subway_stations,
                                                     lat_col='latitude',
                                                     lon_col='longitude',
                                                     grid_df=grid_df,
                                                     grid_id_col='grid_id',
                                                     buffer_radius_m=1000)
subway_buffer_joined

,latitude,longitude,geometry,index_right,grid_id
0,40.7021,-74.0137,POINT (980461.309 195059.95),33,M-0034
0,40.7021,-74.0137,POINT (980461.309 195059.95),36,M-0037
0,40.7021,-74.0137,POINT (980461.309 195059.95),37,M-0038
0,40.7021,-74.0137,POINT (980461.309 195059.95),38,M-0039
0,40.7021,-74.0137,POINT (980461.309 195059.95),43,M-0044
...,...,...,...,...,...
15846,40.7188,-73.9999,POINT (984280.438 201156.018),138,M-0139
26389,40.7547,-73.9868,POINT (987919.501 214224.897),371,M-0372
26389,40.7547,-73.9868,POINT (987919.501 214224.897),372,M-0373
26389,40.7547,-73.9868,POINT (987919.501 214224.897),390,M-0391


In [26]:
subway_grid_df = subway_df.merge(subway_joined[['latitude', 'longitude', 'grid_id']], on=['latitude', 'longitude'])
subway_grid_df = subway_grid_df.groupby(['transit_timestamp', 'grid_id'])['ridership'].sum().reset_index(name='subway_grid_ridership')

subway_buffer_df = subway_df.merge(subway_buffer_joined[['latitude', 'longitude', 'grid_id']], on=['latitude', 'longitude'])
subway_buffer_df = subway_buffer_df.groupby(['transit_timestamp', 'grid_id'])['ridership'].sum().reset_index(name='subway_buffer_ridership')

merged_subway_ridership = subway_grid_df.merge(subway_buffer_df, on=['transit_timestamp', 'grid_id'], how='outer').fillna(0)
merged_subway_ridership['date'] = pd.to_datetime(merged_subway_ridership['transit_timestamp']).dt.date
merged_subway_ridership['hour'] = (pd.to_datetime(merged_subway_ridership['transit_timestamp']).dt.hour).astype('int')

In [27]:
integrated_df_3 = integrated_df_2.merge(merged_subway_ridership, on=['date', 'hour', 'grid_id'], how='left').fillna(0)
integrated_df_3 = integrated_df_3.drop(columns=['transit_timestamp'])
integrated_df_3

,date,hour,temp_c,dew_c,wind_speed_knot,precip_mm,grid_id,lat,lon,geometry,is_holiday,restaurant_count,subway_grid_ridership,subway_buffer_ridership
0,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0001,40.6840,-74.0257,"POLYGON ((-74.0242 40.68292, -74.0242 40.68517...",False,0.0,0.0,0.0
1,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0002,40.6840,-74.0227,"POLYGON ((-74.02126 40.68292, -74.02126 40.685...",False,0.0,0.0,0.0
2,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0003,40.6840,-74.0198,"POLYGON ((-74.01832 40.68292, -74.01832 40.685...",False,0.0,0.0,0.0
3,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0004,40.6863,-74.0257,"POLYGON ((-74.0242 40.68517, -74.0242 40.68742...",False,0.0,0.0,0.0
4,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0005,40.6863,-74.0227,"POLYGON ((-74.02126 40.68517, -74.02126 40.687...",False,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9986395,2025-03-31,23,15.6,13.3,1.5,0.8,M-1136,40.8777,-73.9139,"POLYGON ((-73.91244 40.87661, -73.91244 40.878...",False,0.0,0.0,0.0
9986396,2025-03-31,23,15.6,13.3,1.5,0.8,M-1137,40.8777,-73.9110,"POLYGON ((-73.90949 40.87661, -73.90949 40.878...",False,0.0,0.0,0.0
9986397,2025-03-31,23,15.6,13.3,1.5,0.8,M-1138,40.8777,-73.9080,"POLYGON ((-73.90655 40.87661, -73.90655 40.878...",False,0.0,0.0,0.0
9986398,2025-03-31,23,15.6,13.3,1.5,0.8,M-1139,40.8800,-73.9227,"POLYGON ((-73.92126 40.87886, -73.92126 40.881...",False,0.0,0.0,0.0


### wheelchair_df
Todo: Check how to integrate with other datasets

In [28]:
wheelchair_df

,OBJECTID,RecordID,Name,Address_Line_1,Address_Line_2,Borough,Cuisine
0,2,50126747,"U"" LIKE CHINESE TAKE OUT""",4926 BROADWAY,"New York, NY 10034",Manhattan,Chinese
1,3,50136066,THE ONE CLUB,1 WALL STREET,"New York, NY 10005",Manhattan,Mediterranean
2,4,50018480,"W"" CAFE""",390 5 AVENUE,"New York, NY 10018",Manhattan,American
3,5,50115654,10TH FL FOOD HALL ( DEUTSCHE BANK CAFE),10 COLUMBUS CIRCLE,"New York, NY 10019",Manhattan,American
4,7,50112530,11:11 RESTAURANT LOUNGE,3964 10 AVENUE,"New York, NY 10034",Manhattan,Other
...,...,...,...,...,...,...,...
840,2541,50138090,GRAND SICHUAN EASTERN,1049 2 AVENUE,"New York, NY 10022",Manhattan,Chinese
841,2545,50119992,FIGARO,184 BLEECKER STREET,"New York, NY 10012",Manhattan,American
842,2547,50106327,GRANNY ANNIE'S BAR & KITCHEN,425 MAIN STREET,"New York, NY 10044",Manhattan,Irish
843,2550,50134353,GRAYSON HOTEL / HARTA / BAR CIMA,30 WEST 39 STREET,"New York, NY 10018",Manhattan,American


## Generate weighted criteria

In [29]:
grid_weighted_criteria = grid_df[['grid_id']].merge(restaurant_counts, on='grid_id', how='left').fillna(0)
grid_weighted_criteria['weighted'] = (grid_weighted_criteria['restaurant_count'] // 10).astype('int') + 1
grid_weighted_criteria = grid_weighted_criteria[['grid_id', 'weighted']]
grid_weighted_criteria.sort_values('weighted')

,grid_id,weighted
1134,M-1135,1
33,M-0034,1
34,M-0035,1
35,M-0036,1
36,M-0037,1
...,...,...
196,M-0197,8
318,M-0319,9
187,M-0188,9
416,M-0417,10


## GeoDataFrame

### bike_df

In [30]:
bike_stations = bike_df[['lat', 'lon']].drop_duplicates()
bike_joined = gen_point_area_mapping(point_df=bike_stations,
                                     lat_col='lat',
                                     lon_col='lon',
                                     geometry_df=grid_df[['grid_id', 'geometry']],
                                     geometry_col='geometry')

bike_buffer_joined = gen_point_area_buffer_mapping(point_df=bike_stations,
                                                     lat_col='lat',
                                                     lon_col='lon',
                                                     grid_df=grid_df[['grid_id', 'geometry']],
                                                     grid_id_col='grid_id',
                                                     buffer_radius_m=1000)

display(bike_joined.head())
display(bike_buffer_joined.head())

,lat,lon,geometry,index_right,grid_id
0,40.7012,-74.0123,POINT (-74.01234 40.70122),38,M-0039
1,40.7019,-74.0139,POINT (-74.01394 40.70191),37,M-0038
2,40.7037,-74.0117,POINT (-74.01168 40.70365),44,M-0045
3,40.7034,-74.0079,POINT (-74.00787 40.70337),45,M-0046
4,40.7064,-74.0056,POINT (-74.0056 40.70641),54,M-0055


,lat,lon,geometry,index_right,grid_id
0,40.7012,-74.0123,POINT (980827.771 194750.682),33,M-0034
0,40.7012,-74.0123,POINT (980827.771 194750.682),34,M-0035
0,40.7012,-74.0123,POINT (980827.771 194750.682),37,M-0038
0,40.7012,-74.0123,POINT (980827.771 194750.682),38,M-0039
1,40.7019,-74.0139,POINT (980384.218 195000.576),33,M-0034


In [31]:
bike_grid_df = bike_df.merge(bike_joined[['lat', 'lon', 'grid_id']], on=['lat', 'lon'])
bike_grid_df = bike_grid_df.groupby(['trip_time', 'grid_id'])['bike_trips'].sum().reset_index(name='bike_grid_trips')

bike_buffer_df = bike_df.merge(bike_buffer_joined[['lat', 'lon', 'grid_id']], on=['lat', 'lon'])
bike_buffer_df = bike_buffer_df.groupby(['trip_time', 'grid_id'])['bike_trips'].sum().reset_index(name='bike_buffer_ridership')

merged_bike_trips = bike_grid_df.merge(bike_buffer_df, on=['trip_time', 'grid_id'], how='outer').fillna(0)
merged_bike_trips['date'] = pd.to_datetime(merged_bike_trips['trip_time']).dt.date
merged_bike_trips['hour'] = (pd.to_datetime(merged_bike_trips['trip_time']).dt.hour).astype('int')

In [32]:
integrated_df_4 = integrated_df_3.merge(merged_bike_trips, on=['date', 'hour', 'grid_id'], how='left').fillna(0)
integrated_df_4 = integrated_df_4.drop(columns=['trip_time'])
integrated_df_4.head()

,date,hour,temp_c,dew_c,wind_speed_knot,precip_mm,grid_id,lat,lon,geometry,is_holiday,restaurant_count,subway_grid_ridership,subway_buffer_ridership,bike_grid_trips,bike_buffer_ridership
0,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0001,40.6840,-74.0257,"POLYGON ((-74.0242 40.68292, -74.0242 40.68517...",False,0.0,0.0,0.0,0.0,0.0
1,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0002,40.6840,-74.0227,"POLYGON ((-74.02126 40.68292, -74.02126 40.685...",False,0.0,0.0,0.0,0.0,0.0
2,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0003,40.6840,-74.0198,"POLYGON ((-74.01832 40.68292, -74.01832 40.685...",False,0.0,0.0,0.0,0.0,0.0
3,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0004,40.6863,-74.0257,"POLYGON ((-74.0242 40.68517, -74.0242 40.68742...",False,0.0,0.0,0.0,0.0,0.0
4,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0005,40.6863,-74.0227,"POLYGON ((-74.02126 40.68517, -74.02126 40.687...",False,0.0,0.0,0.0,0.0,0.0


In [33]:
integrated_df_4[integrated_df_4['bike_grid_trips'] > 0].head()

,date,hour,temp_c,dew_c,wind_speed_knot,precip_mm,grid_id,lat,lon,geometry,is_holiday,restaurant_count,subway_grid_ridership,subway_buffer_ridership,bike_grid_trips,bike_buffer_ridership
37,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0038,40.7021,-74.0139,"POLYGON ((-74.01244 40.70093, -74.01244 40.703...",False,13.0,99.0,99.0,2.0,4.0
38,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0039,40.7021,-74.0110,"POLYGON ((-74.00949 40.70093, -74.00949 40.703...",False,20.0,0.0,99.0,1.0,5.0
44,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0045,40.7043,-74.0110,"POLYGON ((-74.00949 40.70319, -74.00949 40.705...",False,71.0,0.0,184.0,1.0,8.0
45,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0046,40.7043,-74.0080,"POLYGON ((-74.00655 40.70319, -74.00655 40.705...",False,16.0,0.0,34.0,1.0,1.0
51,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0052,40.7066,-74.0139,"POLYGON ((-74.01244 40.70544, -74.01244 40.707...",False,12.0,34.0,163.0,6.0,6.0


### taxi_df

In [34]:
# Spatial join - map each taxi zone to intersecting grid(s)
taxi_zone_to_grid = gen_area_area_mapping(zone_df=taxi_zones, 
                                          grid_df=grid_df,
                                          zone_location_id='location_id')

# Merge with grid weight information
weighted_taxi_zone_to_grid = taxi_zone_to_grid.merge(grid_weighted_criteria, on=['grid_id'])

# Calculate the percentage of each grid's contribution to the taxi zone
weighted_taxi_zone_to_grid['location_tot_weighted'] = weighted_taxi_zone_to_grid.groupby('location_id')['weighted'].transform('sum')
weighted_taxi_zone_to_grid['percentage'] = weighted_taxi_zone_to_grid['weighted'] / weighted_taxi_zone_to_grid['location_tot_weighted']

# Distribute passenger counts across grids using weighted share
merged_taxi_df = taxi_df.merge(weighted_taxi_zone_to_grid[['location_id', 'grid_id', 'percentage']], on='location_id')
merged_taxi_df['weighted_passenger_count'] = (merged_taxi_df['passenger_count'] * merged_taxi_df['percentage']).astype('int')
merged_taxi_df = merged_taxi_df.groupby(['trip_time', 'grid_id'])['weighted_passenger_count'].sum().reset_index(name='taxi_passenger_count')
merged_taxi_df.head()

,trip_time,grid_id,taxi_passenger_count
0,2024-01-01 00:00:00,M-0034,5
1,2024-01-01 00:00:00,M-0035,4
2,2024-01-01 00:00:00,M-0036,4
3,2024-01-01 00:00:00,M-0037,1
4,2024-01-01 00:00:00,M-0038,12


In [35]:
merged_taxi_df['date'] = pd.to_datetime(merged_taxi_df['trip_time']).dt.date
merged_taxi_df['hour'] = (pd.to_datetime(merged_taxi_df['trip_time']).dt.hour).astype('int')

integrated_df_5 = integrated_df_4.merge(merged_taxi_df, on=['date', 'hour', 'grid_id'], how='left').fillna(0)
integrated_df_5 = integrated_df_5.drop(columns=['trip_time'])
integrated_df_5.head()

,date,hour,temp_c,dew_c,wind_speed_knot,precip_mm,grid_id,lat,lon,geometry,is_holiday,restaurant_count,subway_grid_ridership,subway_buffer_ridership,bike_grid_trips,bike_buffer_ridership,taxi_passenger_count
0,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0001,40.6840,-74.0257,"POLYGON ((-74.0242 40.68292, -74.0242 40.68517...",False,0.0,0.0,0.0,0.0,0.0,0.0
1,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0002,40.6840,-74.0227,"POLYGON ((-74.02126 40.68292, -74.02126 40.685...",False,0.0,0.0,0.0,0.0,0.0,0.0
2,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0003,40.6840,-74.0198,"POLYGON ((-74.01832 40.68292, -74.01832 40.685...",False,0.0,0.0,0.0,0.0,0.0,0.0
3,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0004,40.6863,-74.0257,"POLYGON ((-74.0242 40.68517, -74.0242 40.68742...",False,0.0,0.0,0.0,0.0,0.0,0.0
4,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0005,40.6863,-74.0227,"POLYGON ((-74.02126 40.68517, -74.02126 40.687...",False,0.0,0.0,0.0,0.0,0.0,0.0


### event_df

In [36]:
event_zone = event_df[['main_location', 'geometry']].drop_duplicates()
event_zone_to_grid = gen_area_area_mapping(zone_df=event_zone, 
                                           grid_df=grid_df,
                                           zone_location_id='main_location')
event_zone_to_grid

,grid_id,geometry,index_right,main_location
33,M-0034,"POLYGON ((-74.01244 40.69868, -74.01244 40.700...",48289,WHITEHALL STREET
33,M-0034,"POLYGON ((-74.01244 40.69868, -74.01244 40.700...",68,WATER STREET
33,M-0034,"POLYGON ((-74.01244 40.69868, -74.01244 40.700...",290,Battery Park
34,M-0035,"POLYGON ((-74.00949 40.69868, -74.00949 40.700...",48289,WHITEHALL STREET
34,M-0035,"POLYGON ((-74.00949 40.69868, -74.00949 40.700...",68,WATER STREET
...,...,...,...,...
1126,M-1127,"POLYGON ((-73.92126 40.87436, -73.92126 40.876...",585,Inwood Hill Park
1127,M-1128,"POLYGON ((-73.91832 40.87436, -73.91832 40.876...",585,Inwood Hill Park
1132,M-1133,"POLYGON ((-73.92714 40.87661, -73.92714 40.878...",585,Inwood Hill Park
1133,M-1134,"POLYGON ((-73.9242 40.87661, -73.9242 40.87886...",585,Inwood Hill Park


In [37]:
merged_event_df = event_df.merge(event_zone_to_grid[['main_location', 'grid_id']], on='main_location')
merged_event_df['start_time'] = pd.to_datetime(merged_event_df['start_time'])
merged_event_df['end_time'] = pd.to_datetime(merged_event_df['end_time'])

expanded_rows = []

for _, row in merged_event_df.iterrows():
    hours = pd.date_range(start=row['start_time'], end=row['end_time'], freq='h')
    for h in hours:
        expanded_rows.append({'event_hour': h, 'grid_id': row['grid_id']})

In [38]:
event_hourly_df = pd.DataFrame(expanded_rows)
event_hourly_df['event_rounded_hour'] = pd.to_datetime(event_hourly_df['event_hour']).dt.floor('h')
event_hourly_count = event_hourly_df.groupby(['event_rounded_hour', 'grid_id']).size().reset_index(name='event_count')
event_hourly_count['date'] = pd.to_datetime(event_hourly_count['event_rounded_hour']).dt.date
event_hourly_count['hour'] = (pd.to_datetime(event_hourly_count['event_rounded_hour']).dt.hour).astype('int')

In [39]:
integrated_df_6 = integrated_df_5.merge(event_hourly_count, on=['date', 'hour', 'grid_id'], how='left').fillna(0)
integrated_df_6 = integrated_df_6.drop(columns=['event_rounded_hour'])
integrated_df_6.head()

,date,hour,temp_c,dew_c,wind_speed_knot,precip_mm,grid_id,lat,lon,geometry,is_holiday,restaurant_count,subway_grid_ridership,subway_buffer_ridership,bike_grid_trips,bike_buffer_ridership,taxi_passenger_count,event_count
0,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0001,40.6840,-74.0257,"POLYGON ((-74.0242 40.68292, -74.0242 40.68517...",False,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0002,40.6840,-74.0227,"POLYGON ((-74.02126 40.68292, -74.02126 40.685...",False,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0003,40.6840,-74.0198,"POLYGON ((-74.01832 40.68292, -74.01832 40.685...",False,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0004,40.6863,-74.0257,"POLYGON ((-74.0242 40.68517, -74.0242 40.68742...",False,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0005,40.6863,-74.0227,"POLYGON ((-74.02126 40.68517, -74.02126 40.687...",False,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [42]:
save_large_dataframe_in_chunks(source_df=integrated_df_6,
                               file_name='../integrated_dataset/integrated_data',
                               chunk_size=1000000)

../integrated_dataset/integrated_data_part1.csv
../integrated_dataset/integrated_data_part2.csv
../integrated_dataset/integrated_data_part3.csv
../integrated_dataset/integrated_data_part4.csv
../integrated_dataset/integrated_data_part5.csv
../integrated_dataset/integrated_data_part6.csv
../integrated_dataset/integrated_data_part7.csv
../integrated_dataset/integrated_data_part8.csv
../integrated_dataset/integrated_data_part9.csv
../integrated_dataset/integrated_data_part10.csv


### population_df

In [43]:
# Convert to correct data type
population_df['population'] = population_df['population'].str.replace(',', '').astype('int')

population_zone_to_grid = gen_area_area_mapping(zone_df=population_df, 
                                                grid_df=grid_df,
                                                zone_location_id='NAMELSAD')

# Merge with grid weight information
weighted_population_zone_to_grid = population_zone_to_grid.merge(grid_weighted_criteria, on=['grid_id'])

# Calculate the percentage of each grid's contribution to the taxi zone
weighted_population_zone_to_grid['location_tot_weighted'] = weighted_population_zone_to_grid.groupby('NAMELSAD')['weighted'].transform('sum')
weighted_population_zone_to_grid['percentage'] = weighted_population_zone_to_grid['weighted'] / weighted_population_zone_to_grid['location_tot_weighted']

# Distribute population across grids using weighted share
merged_population_df = population_df.merge(weighted_population_zone_to_grid[['NAMELSAD', 'grid_id', 'percentage']], on='NAMELSAD')
merged_population_df['weighted_population'] = (merged_population_df['population'] * merged_population_df['percentage']).astype('int')
merged_population_df = merged_population_df.groupby(['grid_id'])['weighted_population'].sum().reset_index(name='population')
merged_population_df.head()

,grid_id,population
0,M-0001,0
1,M-0002,0
2,M-0003,0
3,M-0004,0
4,M-0005,0


In [44]:
integrated_df_7 = integrated_df_6.merge(merged_population_df, on=['grid_id'], how='left').fillna(0)
integrated_df_7.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9986400 entries, 0 to 9986399
Data columns (total 19 columns):
 #   Column                   Dtype   
---  ------                   -----   
 0   date                     object  
 1   hour                     int64   
 2   temp_c                   float64 
 3   dew_c                    float64 
 4   wind_speed_knot          float64 
 5   precip_mm                float64 
 6   grid_id                  object  
 7   lat                      float64 
 8   lon                      float64 
 9   geometry                 geometry
 10  is_holiday               bool    
 11  restaurant_count         float64 
 12  subway_grid_ridership    float64 
 13  subway_buffer_ridership  float64 
 14  bike_grid_trips          float64 
 15  bike_buffer_ridership    float64 
 16  taxi_passenger_count     float64 
 17  event_count              float64 
 18  population               float64 
dtypes: bool(1), float64(14), geometry(1), int64(1), object(2)
memor

## Save integrated dataset

In [45]:
save_large_dataframe_in_chunks(source_df=integrated_df_7,
                               file_name='../integrated_dataset/integrated_data',
                               chunk_size=1000000)

../integrated_dataset/integrated_data_part1.csv
../integrated_dataset/integrated_data_part2.csv
../integrated_dataset/integrated_data_part3.csv
../integrated_dataset/integrated_data_part4.csv
../integrated_dataset/integrated_data_part5.csv
../integrated_dataset/integrated_data_part6.csv
../integrated_dataset/integrated_data_part7.csv
../integrated_dataset/integrated_data_part8.csv
../integrated_dataset/integrated_data_part9.csv
../integrated_dataset/integrated_data_part10.csv


# Load integrated dataset from CSV files
- The integrated dataset is split into multiple CSV files and zipped as `integrated_dataset`.
- Download and unzip from the [shared Google Drive](https://drive.google.com/drive/u/2/folders/1Yr7l0EcolHcL2TDrrq72ZUix2FNEc5X2).

In [47]:
combined_df = load_all_geoDataFrames_in_folder("../integrated_dataset/")

../integrated_dataset\integrated_data_part1.csv
../integrated_dataset\integrated_data_part10.csv
../integrated_dataset\integrated_data_part2.csv
../integrated_dataset\integrated_data_part3.csv
../integrated_dataset\integrated_data_part4.csv
../integrated_dataset\integrated_data_part5.csv
../integrated_dataset\integrated_data_part6.csv
../integrated_dataset\integrated_data_part7.csv
../integrated_dataset\integrated_data_part8.csv
../integrated_dataset\integrated_data_part9.csv


In [48]:
len(combined_df)

9986400

In [ ]:
combined_df = combined_df.sort_values(['date', 'hour'])
combined_df.head()

,date,hour,temp_c,dew_c,wind_speed_knot,precip_mm,grid_id,lat,lon,geometry,is_holiday,restaurant_count,subway_grid_ridership,subway_buffer_ridership,bike_grid_trips,bike_buffer_ridership,taxi_passenger_count,event_count,population
0,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0001,40.6840,-74.0257,"POLYGON ((-74.0242 40.68292, -74.0242 40.68517...",False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0002,40.6840,-74.0227,"POLYGON ((-74.02126 40.68292, -74.02126 40.685...",False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0003,40.6840,-74.0198,"POLYGON ((-74.01832 40.68292, -74.01832 40.685...",False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0004,40.6863,-74.0257,"POLYGON ((-74.0242 40.68517, -74.0242 40.68742...",False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2024-04-01,0,13.3,-0.6,0.0,0.0,M-0005,40.6863,-74.0227,"POLYGON ((-74.02126 40.68517, -74.02126 40.687...",False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Save combined_df to pickle to improve efficiency

In [ ]:
# File has already been saved; this line is now commented out to prevent overwriting
# Check google drive for the file
# combined_df.to_pickle("combined_integrated_df.pkl")